## Environmental Context

This notebook fetches gridded environmental data from [Bio-Oracle](https://erddap.bio-oracle.org/erddap/griddap) and seafloor lithology from the [NOAA/NCEI IMLGS](https://www.ncei.noaa.gov/erddap/tabledap/IMLGS_Marine_Geology.html) for the spatial extent of the coral dataset. Bio-Oracle layers use depth-mean baseline climatologies (2000–2019 or 2000–2018 depending on the variable). Each section fetches one variable, joins it onto the cleaned coral records by nearest grid cell or sample point, and spot-checks coverage where in-situ measurements are available.

Variables fetched:

- **Temperature** (`thetao_mean`) — Bio-Oracle, baseline 2000–2019
- **Salinity** (`so_mean`) — Bio-Oracle, baseline 2000–2019
- **Dissolved oxygen** (`o2_mean`) — Bio-Oracle, baseline 2000–2018
- **pH** (`ph_mean`) — Bio-Oracle, baseline 2000–2018
- **Seafloor lithology** (`lith1`) — IMLGS primary lithology classification

In [1]:
import pandas as pd

from se_deep_coral_communities.data_loading.bio_oracle import (
    BioOracleOxygenSource,
    BioOraclePHSource,
    BioOracleSalinitySource,
    BioOracleTemperatureSource,
)
from se_deep_coral_communities.data_loading.imlgs import IMLGSSubstrateSource

coral_df = pd.read_parquet("../data/processed/se_coral_records_cleaned.parquet")

## Temperature

`thetao_mean` tracks in-situ temperature closely (median difference ~ 1 °C). The gridded value fills gaps where no in-situ measurement was recorded.

In [2]:
joined_df = BioOracleTemperatureSource().fetch_join(coral_df)

print(
    f"Rows: {len(joined_df)}, thetao_mean coverage: {joined_df['thetao_mean'].notna().mean():.1%}"
)
joined_df.head()

Rows: 71499, thetao_mean coverage: 100.0%


,lat,lon,catalog_num,record_type,loc_accuracy,scientific_nm,depth,gis_depth,large_marine_ecosystem,country,...,density,sample_area,temp,salinity,oxygen,substrate,cmecs_substrate,cmecs_geo_form,is_shallow,thetao_mean
0,17.92201,-66.64733,953318,video observation,20m,Pennatuloidea,284.0,276.0,Caribbean Sea,USA,...,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,False,25.755342
1,17.92225,-66.64734,953319,video observation,20m,Pennatuloidea,282.0,276.0,Caribbean Sea,USA,...,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,False,25.755342
2,17.92228,-66.64734,953320,video observation,20m,Pennatuloidea,280.0,276.0,Caribbean Sea,USA,...,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,False,25.755342
3,17.92228,-66.64732,953321,video observation,20m,Pennatuloidea,279.0,276.0,Caribbean Sea,USA,...,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,False,25.755342
4,17.92227,-66.64732,953322,video observation,20m,Pennatuloidea,279.0,276.0,Caribbean Sea,USA,...,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,False,25.755342


In [3]:
temps = joined_df[["temp", "thetao_mean"]].dropna()
temps["diff"] = temps["temp"] - temps["thetao_mean"]
temps

,temp,thetao_mean,diff
2847,4.3,4.095151,0.204849
2848,5.7,5.181584,0.518416
2849,6.2,5.181584,1.018416
2850,7.3,5.181584,2.118416
2851,7.3,5.181584,2.118416
...,...,...,...
71494,18.1,18.237656,-0.137656
71495,18.1,18.237656,-0.137656
71496,17.9,18.237656,-0.337656
71497,17.9,18.237656,-0.337656


## Salinity


In [4]:
salinity_df = BioOracleSalinitySource().fetch_join(coral_df)

print(
    f"Rows: {len(salinity_df)}, so_mean coverage: {salinity_df['so_mean'].notna().mean():.1%}"
)

in_situ = salinity_df[["salinity", "so_mean"]].dropna()
in_situ["diff"] = in_situ["salinity"] - in_situ["so_mean"]
print(
    f"In-situ vs gridded salinity — median diff: {in_situ['diff'].median():.3f} PSU, "
    f"mean abs diff: {in_situ['diff'].abs().mean():.3f} PSU"
)
in_situ.head()

Rows: 71499, so_mean coverage: 100.0%
In-situ vs gridded salinity — median diff: 0.132 PSU, mean abs diff: 0.171 PSU


,salinity,so_mean,diff
2847,35.0,34.963136,0.036864
2848,34.9,34.928592,-0.028592
2849,34.9,34.928592,-0.028592
2850,34.9,34.928592,-0.028592
2851,34.9,34.928592,-0.028592


## Dissolved oxygen


In [5]:
oxygen_df = BioOracleOxygenSource().fetch_join(coral_df)

print(
    f"Rows: {len(oxygen_df)}, o2_mean coverage: {oxygen_df['o2_mean'].notna().mean():.1%}"
)

in_situ = oxygen_df[["oxygen", "o2_mean"]].dropna()
if not in_situ.empty:
    in_situ["diff"] = in_situ["oxygen"] - in_situ["o2_mean"]
    print(
        f"In-situ vs gridded oxygen — median diff: {in_situ['diff'].median():.4f}, "
        f"mean abs diff: {in_situ['diff'].abs().mean():.4f}"
    )
    display(in_situ.head())
else:
    print("No overlapping in-situ oxygen measurements for comparison.")

Rows: 71499, o2_mean coverage: 100.0%
In-situ vs gridded oxygen — median diff: -168.7135, mean abs diff: 172.1228


,oxygen,o2_mean,diff
2847,6.3,203.762223,-197.462223
2848,4.5,173.340942,-168.840942
2849,4.3,173.340942,-169.040942
2850,2.8,173.340942,-170.540942
2851,2.7,173.340942,-170.640942


## pH


In [6]:
ph_df = BioOraclePHSource().fetch_join(coral_df)

print(f"Rows: {len(ph_df)}, ph_mean coverage: {ph_df['ph_mean'].notna().mean():.1%}")
print(f"ph_mean range: {ph_df['ph_mean'].min():.3f} – {ph_df['ph_mean'].max():.3f}")
ph_df["ph_mean"].describe().round(4)

Rows: 71499, ph_mean coverage: 100.0%
ph_mean range: 7.775 – 8.092


count    71499.0000
mean         7.9917
std          0.0620
min          7.7750
25%          7.9422
50%          8.0097
75%          8.0407
max          8.0920
Name: ph_mean, dtype: float64

## Seafloor lithology

`lith1` is the primary lithology classification from the nearest NOAA/NCEI IMLGS geological sample with water depth ≥ 50 m. Unlike the Bio-Oracle layers, this is a categorical variable assigned by nearest-neighbour join to physical dredge and core samples. Distance to the nearest sample varies; treat assignments for isolated events cautiously.

In [7]:
substrate_df = IMLGSSubstrateSource().fetch_join(coral_df)

print(f"Rows: {len(substrate_df)}, lith1 coverage: {substrate_df['lith1'].notna().mean():.1%}")
print(f"\nTop lithology classes:")
print(substrate_df["lith1"].value_counts().head(10))

# Compare against sparse in-situ substrate field where both are present
in_situ = substrate_df[["substrate", "lith1"]].dropna()
if not in_situ.empty:
    print(f"\nIn-situ substrate vs IMLGS lith1 ({len(in_situ)} overlapping records):")
    display(pd.crosstab(in_situ["substrate"], in_situ["lith1"]))

Rows: 71499, lith1 coverage: 100.0%

Top lithology classes:
lith1
terrigenous                 39436
calcareous, foraminifera    21033
calcareous, biogenic         3931
clay minerals                3389
volcanics                    2105
calcareous, shells            666
calcareous, coral             519
calcareous, pteropods         241
manganese                     125
calcareous, nannofossils       44
Name: count, dtype: int64

In-situ substrate vs IMLGS lith1 (3501 overlapping records):


lith1,"calcareous, biogenic","calcareous, foraminifera","calcareous, shells",clay minerals,terrigenous,volcanics
substrate,,,,,,
Algal nodules,0,0,0,327,0,0
Dead Coral Framework,0,0,0,0,1,0
Debris,2,0,0,0,0,0
Deep reef,0,0,0,2099,243,0
Reefs,0,0,0,293,31,0
Rope,0,0,0,0,0,4
Soft,2,0,1,0,0,1
Soft bottom,0,0,0,448,31,0
pipe,2,0,0,0,0,0


## All contextual variables combined

Assemble one table with all five contextual variables joined onto the cleaned coral records.

In [8]:
context_df = coral_df.assign(
    thetao_mean=joined_df["thetao_mean"],
    so_mean=salinity_df["so_mean"],
    o2_mean=oxygen_df["o2_mean"],
    ph_mean=ph_df["ph_mean"],
    lith1=substrate_df["lith1"],
)

print(f"Rows: {len(context_df)}")
print("Coverage:")
for col in ["thetao_mean", "so_mean", "o2_mean", "ph_mean", "lith1"]:
    print(f"  {col}: {context_df[col].notna().mean():.1%}")

context_df[["lat", "lon", "depth", "thetao_mean", "so_mean", "o2_mean", "ph_mean", "lith1"]].head()

Rows: 71499
Coverage:
  thetao_mean: 100.0%
  so_mean: 100.0%
  o2_mean: 100.0%
  ph_mean: 100.0%
  lith1: 100.0%


,lat,lon,depth,thetao_mean,so_mean,o2_mean,ph_mean,lith1
0,17.92201,-66.64733,284.0,25.755342,36.798352,214.92998,8.051992,"calcareous, biogenic"
1,17.92225,-66.64734,282.0,25.755342,36.798352,214.92998,8.051992,"calcareous, biogenic"
2,17.92228,-66.64734,280.0,25.755342,36.798352,214.92998,8.051992,"calcareous, biogenic"
3,17.92228,-66.64732,279.0,25.755342,36.798352,214.92998,8.051992,"calcareous, biogenic"
4,17.92227,-66.64732,279.0,25.755342,36.798352,214.92998,8.051992,"calcareous, biogenic"


In [9]:
out_path = "../data/processed/bio_oracle_context.parquet"
context_df.to_parquet(out_path, index=False)
print(f"Saved {len(context_df):,} rows to {out_path}")

Saved 71,499 rows to ../data/processed/bio_oracle_context.parquet
